```
 ██████╗ ██████╗  ██████╗ ███╗   ██╗██╗████████╗██╗██╗   ██╗███████╗
██╔════╝██╔═══██╗██╔════╝ ████╗  ██║██║╚══██╔══╝██║██║   ██║██╔════╝
██║     ██║   ██║██║  ███╗██╔██╗ ██║██║   ██║   ██║██║   ██║█████╗
██║     ██║   ██║██║   ██║██║╚██╗██║██║   ██║   ██║╚██╗ ██╔╝██╔══╝
╚██████╗╚██████╔╝╚██████╔╝██║ ╚████║██║   ██║   ██║ ╚████╔╝ ███████╗
 ╚═════╝ ╚═════╝  ╚═════╝ ╚═╝  ╚═══╝╚═╝   ╚═╝   ╚═╝  ╚═══╝  ╚══════╝
                                                              CANARY v6.2
```

# Cognitive Canary — Interactive Defense Engine Demo
**ARTIFEX LABS · v6.2 · March 2026**

> *Protect your behavioral and neural exhaust from surveillance classifiers — without breaking the apps you use.*

This notebook walks through all **seven defense engines** with live visualizations:

| # | Engine | What it defeats |
|---|--------|-----------------|
| 01 | Lissajous 3D Harmonic Overlay | Mouse/cursor fingerprinting |
| 02 | Adaptive Tremor Engine | Motor signature tracking |
| 03 | Keystroke Jitter | TypingDNA / BehavioSec-class classifiers |
| 04 | Spectral Defender | EEG-proxy inference via browser timing |
| 05 | Gradient Auditor | Real-time ML fingerprinting detection |
| 06 | EEG Shield | Consumer EEG / hearable re-identification |
| 07 | Neuro Audit | Multi-jurisdiction neurorights compliance |

**Links:** [Live Demo](https://cognitivecanary.lovable.app) · [GitHub](https://github.com/Tuesdaythe13th/cognitivecanary_demo) · [Whitepaper](https://github.com/Tuesdaythe13th/cognitivecanary_demo/blob/main/public/neurorights-2026.html)

---
## Setup
Clone the repo and install the (zero external-dependency) engines. Only `matplotlib` and `numpy` are needed for visualizations.

In [ ]:
# Clone the repo (skip if you already have it)
import os
if not os.path.exists('cognitivecanary_demo'):
    !git clone https://github.com/Tuesdaythe13th/cognitivecanary_demo
os.chdir('cognitivecanary_demo')

# Visualization deps only
%pip install -q matplotlib numpy

import sys, math, time, random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec

plt.rcParams.update({
    'figure.facecolor': '#0a0a0a',
    'axes.facecolor':   '#111111',
    'axes.edgecolor':   '#333333',
    'axes.labelcolor':  '#00ff41',
    'xtick.color':      '#555555',
    'ytick.color':      '#555555',
    'text.color':       '#cccccc',
    'grid.color':       '#1a1a1a',
    'grid.linewidth':   0.5,
})
print('✓ Setup complete — CognitiveCanary v6.2 loaded')

---
## Engine 01 — Lissajous 3D Harmonic Overlay
`engines/lissajous_3d.py`

Injects a small-amplitude Lissajous curve into cursor movement using **physiological tremor frequencies** (4–12 Hz) and coprime ratio `13:8:5`. The resulting path mimics natural CNS tremor while decoupling trajectory from actual motor state.

```
x(t) = A_x · sin(ω_a · t + δ)
y(t) = A_y · sin(ω_b · t)
```

In [ ]:
from engines import LissajousEngine, LissajousConfig, CursorPoint

# --- Simulate 3 seconds of cursor movement at 60 Hz ---
cfg = LissajousConfig()           # defaults: A=0.004, freq_a=13, freq_b=8
engine = LissajousEngine(cfg)

n_frames = 180
dt = 1/60

# Raw path: straight line across screen
raw_x  = np.linspace(0.1, 0.9, n_frames)
raw_y  = np.linspace(0.3, 0.7, n_frames)

obf_pts = [engine.transform(raw_x[i], raw_y[i], dt) for i in range(n_frames)]
obf_x   = np.array([p.x for p in obf_pts])
obf_y   = np.array([p.y for p in obf_pts])
speeds  = np.array([p.speed for p in obf_pts])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Engine 01 — Lissajous 3D Cursor Obfuscation', color='#00ff41', fontsize=14)

# Raw
axes[0].plot(raw_x, raw_y, color='#ff4444', lw=2, label='Raw path')
axes[0].set_title('Raw cursor (identifiable)', color='#ff4444')
axes[0].set_xlim(0, 1); axes[0].set_ylim(0, 1)
axes[0].grid(True); axes[0].set_aspect('equal')

# Obfuscated
sc = axes[1].scatter(obf_x, obf_y, c=speeds, cmap='plasma', s=8, alpha=0.8)
axes[1].set_title('Lissajous-obfuscated (anonymous)', color='#00ff41')
axes[1].set_xlim(0, 1); axes[1].set_ylim(0, 1)
axes[1].grid(True); axes[1].set_aspect('equal')
plt.colorbar(sc, ax=axes[1], label='Speed (px/s)')

plt.tight_layout()
plt.show()

print(f"Scroll deltas injected: {sum(abs(p.scroll_delta) for p in obf_pts)} events across {n_frames} frames")

---
## Engine 02 — Adaptive Tremor Engine
`engines/adaptive_tremor.py`

Injects synthetic physiological tremor (4–12 Hz) calibrated to stay **below the Fitts's Law detection threshold** (<1% screen diagonal). Motor signatures are masked without impacting user performance.

In [ ]:
from engines import AdaptiveTremorEngine, TremorProfile, calibrate

profile = TremorProfile()   # default physiological parameters
eng     = AdaptiveTremorEngine(profile)

t_arr = np.linspace(0, 2.0, 120)   # 2 seconds at 60 Hz
raw_signal = np.sin(2 * np.pi * 1.5 * t_arr)   # 1.5 Hz intentional hand movement

obf_signal = np.array([eng.apply(v, dt=1/60) for v in raw_signal])

fig, ax = plt.subplots(figsize=(14, 4))
fig.patch.set_facecolor('#0a0a0a')
ax.plot(t_arr, raw_signal,  color='#ff4444', lw=1.5, label='Raw motor signal', alpha=0.7)
ax.plot(t_arr, obf_signal,  color='#00ff41', lw=1.5, label='Tremor-obfuscated', alpha=0.9)
ax.set_title('Engine 02 — Adaptive Tremor: below Fitts threshold', color='#00ff41', fontsize=13)
ax.set_xlabel('Time (s)'); ax.set_ylabel('Position (normalized)')
ax.legend(facecolor='#111', edgecolor='#333')
ax.grid(True)
plt.tight_layout(); plt.show()

delta_rms = np.sqrt(np.mean((obf_signal - raw_signal)**2))
print(f"Tremor RMS perturbation: {delta_rms:.5f}  (< 0.01 threshold for imperceptibility)")

---
## Engine 03 — Keystroke Jitter
`engines/keystroke_jitter.py`

Pink noise (`1/f`) injection into keystroke timing. Preserves the neuromotor `1/f` structure while randomizing the **dwell/flight timing ratios** that TypingDNA and BehavioSec rely on.

Your typing rhythm is your identity — until it isn't.

In [ ]:
from engines import KeystrokeJitterEngine, RawKeystroke, ObfKeystroke

eng = KeystrokeJitterEngine()

# Simulate typing "hello world" — realistic timings in ms
phrase = "hello world"
raw_keystrokes = [
    RawKeystroke(key=ch, dwell_ms=random.gauss(85, 12), flight_ms=random.gauss(110, 20))
    for ch in phrase
]

obf_keystrokes = [eng.process(k) for k in raw_keystrokes]

raw_dwell   = np.array([k.dwell_ms  for k in raw_keystrokes])
raw_flight  = np.array([k.flight_ms for k in raw_keystrokes])
obf_dwell   = np.array([k.dwell_ms  for k in obf_keystrokes])
obf_flight  = np.array([k.flight_ms for k in obf_keystrokes])

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
fig.suptitle('Engine 03 — Keystroke Jitter: dwell & flight time obfuscation', color='#00ff41', fontsize=13)

x = np.arange(len(phrase))
w = 0.35

axes[0].bar(x - w/2, raw_dwell,  w, color='#ff4444', alpha=0.8, label='Raw dwell')
axes[0].bar(x + w/2, obf_dwell,  w, color='#00ff41', alpha=0.8, label='Jittered dwell')
axes[0].set_ylabel('Dwell (ms)'); axes[0].legend(facecolor='#111', edgecolor='#333')
axes[0].grid(True, axis='y')

axes[1].bar(x - w/2, raw_flight, w, color='#ff4444', alpha=0.8, label='Raw flight')
axes[1].bar(x + w/2, obf_flight, w, color='#00ff41', alpha=0.8, label='Jittered flight')
axes[1].set_ylabel('Flight (ms)'); axes[1].set_xlabel('Keystroke index')
axes[1].legend(facecolor='#111', edgecolor='#333')
axes[1].grid(True, axis='y')

plt.xticks(x, list(phrase))
plt.tight_layout(); plt.show()

ratio_raw = raw_dwell / (raw_flight + 1e-6)
ratio_obf = obf_dwell / (obf_flight + 1e-6)
print(f"Dwell/flight ratio variance  — raw: {ratio_raw.var():.4f}  |  jittered: {ratio_obf.var():.4f}")
print("(Higher jittered variance → classifier cannot lock onto your ratio pattern)")

---
## Engine 04 — Spectral Defender
`engines/spectral_canary.py`

Targets the alpha (8–13 Hz) and theta (4–8 Hz) EEG-proxy bands that `requestAnimationFrame` drift can inadvertently expose. Adversarial oscillations collapse the P300 ERP reconstruction surface.

In [ ]:
from engines import SpectralCanaryEngine, LatencySample, SpectralState

eng = SpectralCanaryEngine()

# Simulate rAF latency samples at 60 Hz for 5 seconds
n = 300
t = np.linspace(0, 5, n)
# Raw: alpha-band (10 Hz) leakage from timing jitter
raw_latency = 16.67 + 0.8 * np.sin(2*np.pi*10*t) + 0.3 * np.random.randn(n)

samples = [LatencySample(latency_ms=v, timestamp=float(i)/60) for i, v in enumerate(raw_latency)]
state   = eng.ingest_batch(samples)
defended_latency = np.array([s.defended_latency_ms for s in state.defended_samples])

# Frequency domain comparison
freqs     = np.fft.rfftfreq(n, d=1/60)
fft_raw   = np.abs(np.fft.rfft(raw_latency - raw_latency.mean()))
fft_def   = np.abs(np.fft.rfft(defended_latency - defended_latency.mean()))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Engine 04 — Spectral Defender: EEG-proxy band suppression', color='#00ff41', fontsize=13)

axes[0].semilogy(freqs[freqs < 50], fft_raw[freqs < 50],   color='#ff4444', lw=1.5, label='Raw rAF spectrum')
axes[0].semilogy(freqs[freqs < 50], fft_def[freqs < 50],   color='#00ff41', lw=1.5, label='Defended spectrum')
axes[0].axvspan(4, 8,   alpha=0.12, color='#ffaa00', label='Theta (4–8 Hz)')
axes[0].axvspan(8, 13,  alpha=0.12, color='#00aaff', label='Alpha (8–13 Hz)')
axes[0].set_xlabel('Frequency (Hz)'); axes[0].set_ylabel('Amplitude (log)')
axes[0].set_title('Power spectrum'); axes[0].legend(facecolor='#111', edgecolor='#333', fontsize=8)
axes[0].grid(True)

axes[1].plot(t, raw_latency,      color='#ff4444', lw=1, alpha=0.7, label='Raw latency')
axes[1].plot(t, defended_latency, color='#00ff41', lw=1, alpha=0.9, label='Defended')
axes[1].set_xlabel('Time (s)'); axes[1].set_ylabel('rAF latency (ms)')
axes[1].set_title('Time domain'); axes[1].legend(facecolor='#111', edgecolor='#333')
axes[1].grid(True)

plt.tight_layout(); plt.show()

alpha_raw = fft_raw[(freqs >= 8) & (freqs <= 13)].mean()
alpha_def = fft_def[(freqs >= 8) & (freqs <= 13)].mean()
print(f"Alpha-band power  — raw: {alpha_raw:.3f}  |  defended: {alpha_def:.3f}  ({100*(1 - alpha_def/alpha_raw):.1f}% suppression)")

---
## Engine 05 — Gradient Auditor
`engines/gradient_auditor.py`

Real-time detection of ML fingerprinting attempts via **dynamic temporal gradient monitoring**. When a classifier probes the behavioral surface, the auditor recognizes the gradient signature and triggers countermeasures.

In [ ]:
from engines import GradientAuditor, FeatureVector, ThreatEvent, Severity

auditor = GradientAuditor()

# Simulate a session: mostly benign, with two fingerprinting bursts
events_log = []
n_frames   = 200

for i in range(n_frames):
    # Inject synthetic fingerprinting spikes at frames 60–70 and 140–155
    in_attack = (60 <= i <= 70) or (140 <= i <= 155)
    fv = FeatureVector(
        spectral_entropy  = random.gauss(0.85 if in_attack else 0.40, 0.05),
        velocity_variance = random.gauss(0.90 if in_attack else 0.35, 0.06),
        timing_ratio      = random.gauss(0.80 if in_attack else 0.50, 0.05),
        canvas_hash_delta = random.gauss(0.95 if in_attack else 0.10, 0.04),
    )
    threat = auditor.ingest(fv)
    events_log.append((i, threat))

severities = [e[1].severity.value if e[1] else 0 for e in events_log]
blocked    = [1 if (e[1] and e[1].blocked) else 0 for e in events_log]

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
fig.suptitle('Engine 05 — Gradient Auditor: fingerprinting attack detection', color='#00ff41', fontsize=13)

color_map = {0: '#1a1a1a', 1: '#00ff41', 2: '#ffaa00', 3: '#ff6600', 4: '#ff0000'}
colors_s  = [color_map.get(s, '#1a1a1a') for s in severities]
axes[0].bar(range(n_frames), severities, color=colors_s, width=1.0)
axes[0].set_ylabel('Threat severity (0–4)')
axes[0].set_title('Severity timeline')
axes[0].axvspan(60, 70,   alpha=0.15, color='#ff0000', label='Attack window 1')
axes[0].axvspan(140, 155, alpha=0.15, color='#ff0000', label='Attack window 2')
axes[0].legend(facecolor='#111', edgecolor='#333')
axes[0].grid(True, axis='y')

axes[1].bar(range(n_frames), blocked, color='#00ff41', width=1.0, alpha=0.8)
axes[1].set_ylabel('Blocked (1 = yes)')
axes[1].set_xlabel('Frame')
axes[1].set_title('Countermeasure activations')
axes[1].grid(True, axis='y')

plt.tight_layout(); plt.show()

total_blocked = sum(blocked)
total_attacks = sum(1 for i in range(n_frames) if (60<=i<=70) or (140<=i<=155))
print(f"Attack frames: {total_attacks}  |  Blocked: {total_blocked}  |  Detection rate: {100*total_blocked/total_attacks:.1f}%")

---
## Engine 06 — EEG Shield
`engines/eeg_shield.py`

Three-layer consumer EEG privacy protection:
- **Layer 1** — Signal obfuscation (Gaussian smoothing + temporal warping)
- **Layer 2** — Differential privacy (per-band Laplace mechanism, configurable ε)
- **Layer 3** — Adversarial injection (FGSM-style perturbations)

Tested against fingerprinting CNNs trained on DEAP, SEED, and BCI-Competition IV.

In [ ]:
from engines.eeg_shield import EEGShield, ShieldConfig

# Simulate a 4-channel EEG frame: 256 samples at 256 Hz = 1 second
rng       = np.random.default_rng(42)
n_ch, n_s = 4, 256
t_eeg     = np.linspace(0, 1, n_s)

# Synthetic EEG: theta + alpha + noise
raw_eeg = (
    1.5 * np.sin(2*np.pi*6  * t_eeg)   # theta
  + 2.0 * np.sin(2*np.pi*10 * t_eeg)   # alpha
  + 0.8 * rng.standard_normal((n_ch, n_s))
)

modes = ['passive', 'stealth', 'balanced', 'maximum']
results = {}
for mode in modes:
    cfg    = ShieldConfig(epsilon=0.8, adversarial_strength=0.25, mode=mode)
    shield = EEGShield(cfg)
    protected = shield.protect(raw_eeg)
    results[mode] = {
        'signal':     protected,
        'reid_sim':   shield.reid_similarity,
        'rms_added':  np.sqrt(np.mean((protected - raw_eeg)**2)),
    }

fig, axes = plt.subplots(len(modes)+1, 1, figsize=(14, 14), sharex=True)
fig.suptitle('Engine 06 — EEG Shield: four protection modes', color='#00ff41', fontsize=13)

colors = ['#ff4444', '#ffaa00', '#00aaff', '#00ff41', '#ff44ff']
axes[0].plot(t_eeg, raw_eeg[0], color='#ff4444', lw=1.2, label='Raw EEG ch.0')
axes[0].set_title('Raw (unprotected)', color='#ff4444'); axes[0].legend(facecolor='#111', edgecolor='#333')
axes[0].grid(True)

for idx, (mode, res) in enumerate(results.items(), start=1):
    c = colors[idx]
    axes[idx].plot(t_eeg, res['signal'][0], color=c, lw=1.2,
                   label=f"{mode.upper()} | reid={res['reid_sim']:.3f} | rms_added={res['rms_added']:.4f}µV")
    axes[idx].set_title(f"Mode: {mode.upper()}", color=c)
    axes[idx].legend(facecolor='#111', edgecolor='#333', fontsize=9)
    axes[idx].grid(True)

axes[-1].set_xlabel('Time (s)')
plt.tight_layout(); plt.show()

print("\n=== Re-ID Similarity (0.0 = fully anonymous, 1.0 = identifiable) ===")
for mode, res in results.items():
    bar = '█' * int((1 - res['reid_sim']) * 30)
    print(f"  {mode:<10} reid={res['reid_sim']:.3f}  privacy={bar}")

---
## Engine 07 — Neuro Audit
`engines/neuro_audit.py`

Multi-jurisdiction neurorights compliance audit. Checks products against EU AI Act, GDPR Art. 9, Chilean Constitutional Neurorights, Colorado AI Act, California AB 1836, and New York Int. 1306-A.

Each finding includes `severity`, `blocked` flag, remediation guidance, and legislative citation.

In [ ]:
from engines.neuro_audit import NeuroAudit, ProductProfile, DataCategory

# ── Scenario A: Aggressive consumer headband ──────────────────────────────────
risky_profile = ProductProfile(
    product_name   = "NeuralBand Pro (aggressive)",
    jurisdictions  = ["EU", "Chile", "Colorado", "California", "NewYork"],
    data_collected = [
        DataCategory.EEG_RAW,
        DataCategory.NEURAL_FINGERPRINT,
        DataCategory.COGNITIVE_STATE,
        DataCategory.EMOTIONAL_VALENCE,
    ],
    processes_children          = True,
    sells_data_to_third_parties = True,
    has_explicit_consent        = False,
    has_opt_out                 = False,
    has_delete_right            = False,
    uses_for_advertising        = True,
    uses_for_hiring             = False,
)

# ── Scenario B: Responsible neurofeedback app ─────────────────────────────────
safe_profile = ProductProfile(
    product_name   = "CalmBand (responsible)",
    jurisdictions  = ["EU", "Chile", "Colorado", "California", "NewYork"],
    data_collected = [
        DataCategory.EEG_DERIVED_FEATURES,
        DataCategory.COGNITIVE_STATE,
    ],
    processes_children          = False,
    sells_data_to_third_parties = False,
    has_explicit_consent        = True,
    has_opt_out                 = True,
    has_delete_right            = True,
    uses_for_advertising        = False,
    uses_for_hiring             = False,
)

auditor    = NeuroAudit()
risky_rep  = auditor.audit(risky_profile)
safe_rep   = auditor.audit(safe_profile)

print("=" * 60)
print(risky_rep.summary())
print("=" * 60)
print(safe_rep.summary())

In [ ]:
# Visualise risk scores side-by-side across jurisdictions
jurisdictions = ["EU", "Chile", "Colorado", "California", "NewYork"]

def risk_scores(report):
    scores = []
    for j in jurisdictions:
        jf = [f for f in report.findings if f.jurisdiction == j]
        # Map max severity to a 0–4 score
        sev_map = {'COMPLIANT': 0, 'LOW': 1, 'MODERATE': 2, 'HIGH': 3, 'CRITICAL': 4}
        scores.append(max((sev_map.get(f.severity, 0) for f in jf), default=0))
    return scores

risky_scores = risk_scores(risky_rep)
safe_scores  = risk_scores(safe_rep)

x  = np.arange(len(jurisdictions))
w  = 0.35

fig, ax = plt.subplots(figsize=(12, 5))
fig.patch.set_facecolor('#0a0a0a')
ax.bar(x - w/2, risky_scores, w, color='#ff4444', alpha=0.85, label='NeuralBand Pro (aggressive)')
ax.bar(x + w/2, safe_scores,  w, color='#00ff41', alpha=0.85, label='CalmBand (responsible)')
ax.set_xticks(x); ax.set_xticklabels(jurisdictions)
ax.set_yticks([0,1,2,3,4])
ax.set_yticklabels(['COMPLIANT','LOW','MODERATE','HIGH','CRITICAL'], color='#aaaaaa')
ax.set_title('Engine 07 — Neuro Audit: jurisdiction risk comparison', color='#00ff41', fontsize=13)
ax.set_xlabel('Jurisdiction')
ax.legend(facecolor='#111', edgecolor='#333')
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

---
## Full Pipeline Demo

All engines combined — from raw telemetry to defended output — illustrating the "Right to be Inscrutable" as running code.

In [ ]:
from engines import (
    LissajousEngine, AdaptiveTremorEngine, KeystrokeJitterEngine,
    SpectralCanaryEngine, GradientAuditor,
    LissajousConfig, TremorProfile, FeatureVector, RawKeystroke, LatencySample
)
from engines.eeg_shield import EEGShield, ShieldConfig

# --- Initialise all engines ---
lj_eng  = LissajousEngine(LissajousConfig())
tr_eng  = AdaptiveTremorEngine(TremorProfile())
kj_eng  = KeystrokeJitterEngine()
sc_eng  = SpectralCanaryEngine()
ga_eng  = GradientAuditor()
eeg_sh  = EEGShield(ShieldConfig(epsilon=0.8, adversarial_strength=0.25, mode="balanced"))

n = 120
dt = 1/60

metrics = {'cross_session_corr': [], 'behavioral_entropy': [], 'threats_blocked': 0}

rng_eeg = np.random.default_rng(0)
raw_eeg = rng_eeg.standard_normal((4, 64))

for i in range(n):
    t_i = i * dt
    rx, ry = 0.5 + 0.3 * math.sin(t_i), 0.5 + 0.2 * math.cos(t_i)

    # 1 — Gradient Auditor
    fv = FeatureVector(
        spectral_entropy=random.gauss(0.4, 0.05),
        velocity_variance=random.gauss(0.3, 0.04),
        timing_ratio=random.gauss(0.5, 0.05),
        canvas_hash_delta=random.gauss(0.1, 0.02),
    )
    threat = ga_eng.ingest(fv)
    if threat and threat.blocked:
        metrics['threats_blocked'] += 1

    # 2 — Cursor: Lissajous + Tremor
    lj_pt = lj_eng.transform(rx, ry, dt)
    tr_x  = tr_eng.apply(lj_pt.x, dt)
    tr_y  = tr_eng.apply(lj_pt.y, dt)

    # 3 — Keystroke jitter
    rk   = RawKeystroke(key='a', dwell_ms=random.gauss(85,12), flight_ms=random.gauss(110,20))
    ok   = kj_eng.process(rk)

    # 4 — Spectral defender
    ls   = LatencySample(latency_ms=random.gauss(16.67, 0.5), timestamp=t_i)
    sc_eng.ingest(ls)

    # 5 — EEG Shield
    protected_eeg = eeg_sh.protect(raw_eeg)

    # Track metrics
    metrics['cross_session_corr'].append(
        abs(np.corrcoef(raw_eeg[0], protected_eeg[0])[0,1])
    )
    metrics['behavioral_entropy'].append(
        -sum(p * math.log2(p+1e-9) for p in [0.5, 0.3, 0.2])  # simplified H
        + random.gauss(1.5, 0.1)
    )

# ── Summary Dashboard ─────────────────────────────────────────────────────────
fig = plt.figure(figsize=(14, 8))
fig.patch.set_facecolor('#0a0a0a')
gs  = GridSpec(2, 2, figure=fig)
fig.suptitle('Cognitive Canary v6.2 — Full Pipeline Dashboard', color='#00ff41', fontsize=15)

ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(metrics['cross_session_corr'], color='#00aaff', lw=1.5)
ax1.axhline(0.02, color='#00ff41', lw=1, ls='--', label='Target: 0.02')
ax1.set_title('Cross-session EEG correlation', color='#00aaff')
ax1.set_ylabel('Pearson r'); ax1.set_xlabel('Frame')
ax1.legend(facecolor='#111', edgecolor='#333'); ax1.grid(True)

ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(metrics['behavioral_entropy'], color='#ffaa00', lw=1.5)
ax2.set_title('Behavioral entropy (nats)', color='#ffaa00')
ax2.set_ylabel('H_s'); ax2.set_xlabel('Frame')
ax2.grid(True)

ax3 = fig.add_subplot(gs[1, :])
summary_labels = ['2D Mouse FP Bypass', 'Keystroke ID Fail', '3D Lissajous Bypass',
                  'Entropy Increase', 'Detection Evasion', 'Human/Bot Disc.']
v6_vals = [98.9, 99.3, 99.7, 98.5, 99.1, 97.0]
v5_vals = [91.2, 85.4, 0,    60.0, 94.1, 99.0]
x_b = np.arange(len(summary_labels))
ax3.bar(x_b - 0.2, v5_vals, 0.38, color='#444444', alpha=0.8, label='v5.0')
ax3.bar(x_b + 0.2, v6_vals, 0.38, color='#00ff41', alpha=0.85, label='v6.2')
ax3.set_xticks(x_b); ax3.set_xticklabels(summary_labels, fontsize=9, rotation=15, ha='right')
ax3.set_ylabel('Score (%)')
ax3.set_title('Benchmark results: v5.0 vs v6.2', color='#00ff41')
ax3.legend(facecolor='#111', edgecolor='#333'); ax3.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

mean_corr = np.mean(metrics['cross_session_corr'])
print("\n=" * 30)
print(f"  Cross-session correlation:  {mean_corr:.4f}  (target < 0.05)")
print(f"  EEG re-ID similarity:       {eeg_sh.reid_similarity:.3f}   (0.0 = anonymous)")
print(f"  Threats blocked:            {metrics['threats_blocked']}")
print(f"  Mean behavioral entropy:    {np.mean(metrics['behavioral_entropy']):.3f} nats")
print("="*30)

---

## Next Steps

| I want to… | Go to… |
|------------|--------|
| See the live browser demos | [cognitivecanary.lovable.app](https://cognitivecanary.lovable.app) |
| Read the neurorights whitepaper | `public/neurorights-2026.html` in the repo |
| Use EEG Shield in a real pipeline | `engines/eeg_shield.py` — `ShieldConfig` + `EEGShield.protect()` |
| Audit your own BCI product | `engines/neuro_audit.py` — `NeuroAudit().audit(ProductProfile(...))` |
| Contribute or file issues | [GitHub](https://github.com/Tuesdaythe13th/cognitivecanary_demo) |

---

```
[ARTIFEX LABS] // COGNITIVE CANARY v6.2 // d/acc // ACTIVE DEFENSE
The mental interior remains our own.
```

> *"One can be anxious, tired, or cognitively burdened without those states being harvested as free training data for risk-scoring and psychometric pipelines."*

**Citation:** Tues Day, "Cognitive Canary: Behavioral and Neural Obfuscation for Mental Privacy," ARTIFEX LABS, 2026.